# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [165]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from joblib import dump
from sklearn.pipeline import Pipeline
from joblib import dump

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [166]:
class FeatureExtractor():
    def __init__(self):
        pass
    
    def fit(self, df, y=None):
        return self
    
    def transform(self, df):
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour
        df['weekday'] = df['timestamp'].dt.dayofweek
        df = df.drop('timestamp', axis=1)
        return df
        

In [167]:
## Apply of FeatureExtractor
df = pd.read_csv('../data/checker_submits.csv')
feature_extractor = FeatureExtractor()
transformed_df = feature_extractor.transform(df)
transformed_df

,uid,labname,numTrials,hour,weekday
0,user_4,project1,1,5,4
1,user_4,project1,2,5,4
2,user_4,project1,3,5,4
3,user_4,project1,4,5,4
4,user_4,project1,5,5,4
...,...,...,...,...,...
1681,user_19,laba06s,9,20,3
1682,user_1,laba06s,6,20,3
1683,user_1,laba06s,7,20,3
1684,user_1,laba06s,8,20,3


In [168]:
class MyOneHotEncoder():
    def __init__(self, target_column=None):
        self.target_column = target_column
        self.encoder = OneHotEncoder()
        
    def fit(self, df, y=None):
        self.categorical_features = df.select_dtypes(include=['object']).columns.tolist()
        if self.target_column in self.categorical_features:
            self.categorical_features.remove(self.target_column)
        self.encoder.fit(df[self.categorical_features])
        return self
    
    def transform(self, df):
        encoded_arr = self.encoder.transform(df[self.categorical_features]).toarray()
        encoded_df = pd.DataFrame(encoded_arr, columns=self.encoder.get_feature_names_out(self.categorical_features), index=df.index)
        df = pd.concat([df, encoded_df], axis=1)
        df = df.drop(columns=self.categorical_features)
        return df
        
        

In [169]:
## Apply of MyOneHotEncoder
my_encoder = MyOneHotEncoder()
my_encoder.fit(transformed_df)
encoded_df = my_encoder.transform(transformed_df)
encoded_df

,numTrials,hour,weekday,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [172]:
class TrainValidationTest():
    def __init__(self):
        pass
    
    def split(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
        X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)
        return X_train, X_valid, X_test, y_train, y_valid, y_test

In [174]:
## Apply of TrainValidationTest
my_split = TrainValidationTest()
X = encoded_df.drop('weekday', axis=1)
y = encoded_df['weekday']
X_train, X_valid, X_test, y_train, y_valid, y_test = my_split.split(X, y)

In [175]:
X_valid

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
1053,1,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1377,30,15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
459,48,11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
132,14,21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1235,25,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
927,1,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1580,26,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
388,3,11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1564,22,21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.772727
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.801484
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.855288
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [176]:
class ModelSelection():
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models = {}
        
    def choose(self, X_train, y_train, X_valid, y_valid):
        for idx, grid in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            print(f"Estimator: {model_name}")

            # Запускаем GridSearchCV
            grid.fit(X_train, y_train)

            # Получаем лучшие параметры и оценку
            best_params = grid.best_params_
            best_score = grid.best_score_

            # Оценка на валидационном наборе
            valid_score = grid.score(X_valid, y_valid)

            # Сохраняем результаты
            self.best_models[model_name] = {
                'params': best_params,
                'valid_score': valid_score,
                'best_model': grid.best_estimator_
            }

            # Используем tqdm для отображения прогресса
            with tqdm(total=len(grid.cv_results_['params']), desc=model_name) as pbar:
                pbar.update(len(grid.cv_results_['params']))

            # Печатаем результаты
            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}\n")

        # Определяем модель с наилучшей валидационной оценкой
        best_model_name = max(self.best_models, key=lambda x: self.best_models[x]['valid_score'])
        print(f"Classifier with best validation set accuracy: {best_model_name}")
        return best_model_name


    def best_results(self):
        results_list = []
        for model_name, metrics in self.best_models.items():
            results_list.append({
                'model': model_name,
                'params': metrics['params'],
                'valid_score': metrics['valid_score']
            })
        return pd.DataFrame(results_list)

In [177]:
# Пример параметров для SVM
svm_params = [{
    'kernel': ['linear', 'rbf'],
    'random_state': [21],
    'probability': [True]
}]

# Создание экземпляра GridSearchCV для SVM
gs_svm = GridSearchCV(estimator=SVC(), param_grid=svm_params, scoring='accuracy', cv=2)

# Пример параметров для других моделей (Decision Tree и Random Forest)
tree_params = {'max_depth': [None, 10, 20], 'class_weight': ['balanced', None]}
rf_params = {'n_estimators': [10, 50], 'max_depth': [None, 10]}

# Создание экземпляров GridSearchCV для других моделей
gs_tree = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=tree_params, scoring='accuracy', cv=2)
gs_rf = GridSearchCV(estimator=RandomForestClassifier(), param_grid=rf_params, scoring='accuracy', cv=2)

# Список всех экземпляров GridSearchCV
grids = [gs_svm, gs_tree, gs_rf]

# Словарь с именами моделей
grid_dict = {0: "SVM", 1: "Decision Tree", 2: "Random Forest"}

# Создание экземпляра ModelSelection
model_selector = ModelSelection(grids, grid_dict)

# Выбор лучшей модели на валидационном наборе (предполагается наличие X_train, y_train, X_valid, y_valid)
best_model_name = model_selector.choose(X_train, y_train, X_valid, y_valid)

# Получение лучших результатов
results_df = model_selector.best_results()
print(results_df)

Estimator: SVM


SVM: 100%|███████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 2652.94it/s]


Best params: {'kernel': 'linear', 'probability': True, 'random_state': 21}
Best training accuracy: 0.633
Validation set accuracy score for best params: 0.607

Estimator: Decision Tree


Decision Tree: 100%|████████████████████████████████████████████████████| 6/6 [00:00<00:00, 29641.72it/s]


Best params: {'class_weight': None, 'max_depth': None}
Best training accuracy: 0.807
Validation set accuracy score for best params: 0.867

Estimator: Random Forest


Random Forest: 100%|████████████████████████████████████████████████████| 4/4 [00:00<00:00, 19463.13it/s]

Best params: {'max_depth': None, 'n_estimators': 50}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.896

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'kernel': 'linear', 'probability': True, 'ran...   
1  Decision Tree          {'class_weight': None, 'max_depth': None}   
2  Random Forest            {'max_depth': None, 'n_estimators': 50}   

   valid_score  
0     0.607407  
1     0.866667  
2     0.896296  


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [178]:
class Finalize():
    def __init__(self, estimator):
        self.estimator = estimator
        
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        return f'Accuracy of the final model is {accuracy}'
    
    def save_model(self, path):
        dump(self.estimator, path)
        print('The model was successfully saved')

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [179]:
df = pd.read_csv('../data/checker_submits.csv')

In [180]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('dayofweek'))
])

In [181]:
data = preprocessing.fit_transform(df)

In [182]:
data

,numTrials,hour,weekday,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [183]:
spliter = TrainValidationTest()
X = data.drop('weekday', axis=1)
y = data['weekday']
X_train, X_valid, X_test, y_train, y_valid, y_test = spliter.split(X, y)

In [184]:
model_selector = ModelSelection(grids, grid_dict)
best_model_name = model_selector.choose(X_train, y_train, X_valid, y_valid)
results_df = pd.DataFrame(model_selector.best_results())
print(results_df)

Estimator: SVM


SVM: 100%|██████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 13025.79it/s]


Best params: {'kernel': 'linear', 'probability': True, 'random_state': 21}
Best training accuracy: 0.633
Validation set accuracy score for best params: 0.607

Estimator: Decision Tree


Decision Tree: 100%|████████████████████████████████████████████████████| 6/6 [00:00<00:00, 23431.87it/s]


Best params: {'class_weight': None, 'max_depth': None}
Best training accuracy: 0.808
Validation set accuracy score for best params: 0.856

Estimator: Random Forest


Random Forest: 100%|████████████████████████████████████████████████████| 4/4 [00:00<00:00, 22162.77it/s]

Best params: {'max_depth': None, 'n_estimators': 50}
Best training accuracy: 0.856
Validation set accuracy score for best params: 0.904

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'kernel': 'linear', 'probability': True, 'ran...   
1  Decision Tree          {'class_weight': None, 'max_depth': None}   
2  Random Forest            {'max_depth': None, 'n_estimators': 50}   

   valid_score  
0     0.607407  
1     0.855556  
2     0.903704  


In [185]:
best_model = model_selector.best_models[best_model_name]['best_model']
finalizer = Finalize(best_model)

# Оценка финальной модели
final_score = finalizer.final_score(X_train, y_train, X_test, y_test)
print(final_score)

Accuracy of the final model is 0.908284023668639


In [186]:
model_filename = f'{best_model_name}_{final_score.split()[-1]}.sav'
finalizer.save_model(model_filename)

The model was successfully saved
